# PS-S06E06: Model Stacking

This notebook tackles the [**Playground Series – Season 6, Episode 6: Predicting Stellar Class**](https://www.kaggle.com/competitions/playground-series-s6e6), a competition focused on predicting a class label (`GALAXY`, `QSO`, `STAR`) for the `class` target, based on features describing the position, light intensity for various spectrums, and other features of the observation.

## Ensemble Strategy: Stacking with a Logistic Meta-Model

Blending (weighted averaging) can fail when one model is dominant or when models are highly correlated. Stacking treats ensembling as a supervised learning problem: we train a small meta-model to combine base model probabilities.

A practical advantage is that a weaker model can still help if it is *differently wrong*—the meta-model can learn when to trust it.


## Install Needed Packages

In [1]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import cross_val_predict, StratifiedKFold

from ps_s06e06_experiment_setup import ExperimentSetup

warnings.filterwarnings('ignore')

%matplotlib inline

In [2]:
helper = ExperimentSetup()

seed = helper.set_seeds()

helper.configure_pandas()
helper.suppress_warnings()

TARGET = 'class'

Random seed set to: 10301
Warnings suppressed.


In [3]:
# Load ground truth labels
training_df = helper.read_dataset('training')

# Encode target labels
target_mapping = {'QSO': 0, 'STAR': 1, 'GALAXY': 2}
y_true = training_df[TARGET].map(target_mapping).astype(int)

TRAINING DATASET

   id   alpha  delta      u      g      r      i      z  redshift  \
0   0 147.734 16.959 25.472 21.896 20.358 19.257 18.621     0.409   
1   1 127.989 32.347 20.779 19.087 17.587 17.226 16.786     0.158   
2   2 179.793 35.345 21.035 21.079 21.172 20.583 20.557     2.824   
3   3 225.818 48.569 23.305 21.051 19.018 18.366 17.915     0.536   
4   4 141.836 19.343 21.703 19.472 18.234 17.899 17.616     0.556   

  spectral_type galaxy_population   class  
0             M      Red_Sequence  GALAXY  
1             M      Red_Sequence  GALAXY  
2           O/B        Blue_Cloud     QSO  
3             M      Red_Sequence  GALAXY  
4             M      Red_Sequence  GALAXY  


In [4]:
submission_df = helper.read_dataset('submission')

## Loading OOF and Test Probabilities

We load the out-of-fold (OOF) and test-set **probabilities** generated by individual model notebooks.

Expected file naming:
- `*_oof_probs.csv` contains columns: `id`, one probability column (e.g. `prob_xgb`), and optionally `target`
- `*_test_probs.csv` contains columns: `id` and one probability column

All files should share the same `id` values as the competition datasets.


In [5]:
oof_files = []
test_files = []

# Local convention: predictions stored under predictions/s06e06/
# Kaggle convention: mount dataset input that contains predictions/
if helper.running_in_kaggle():
    pred_dir = '/kaggle/input/notebooks/stephentarter/ps-s06e06-*/predictions'
else:
    pred_dir = 'predictions'

oof_files = sorted(glob.glob(f'{pred_dir}/*_oof_probs.csv'))
test_files = sorted(glob.glob(f'{pred_dir}/*_test_probs.csv'))

print(f'Found {len(oof_files)} OOF files and {len(test_files)} Test files.')
if len(oof_files) > 0:
    print('OOF files:')
    for f in oof_files:
        print(' -', os.path.basename(f))
if len(test_files) > 0:
    print('Test files:')
    for f in test_files:
        print(' -', os.path.basename(f))

Found 4 OOF files and 4 Test files.
OOF files:
 - catboost_oof_probs.csv
 - lgb_oof_probs.csv
 - nn_tabular_resnet_oof_probs.csv
 - xgb_oof_probs.csv
Test files:
 - catboost_test_probs.csv
 - lgb_test_probs.csv
 - nn_tabular_resnet_test_probs.csv
 - xgb_test_probs.csv


In [6]:
# Helper to load and merge probabilities
def load_probs(file_list, index_col='id'):
    df_list = []
    for file in file_list:
        base = os.path.basename(file)
        model_name = (
            base.replace('_oof_probs.csv', '')
                .replace('_test_probs.csv', '')
        )

        df = pd.read_csv(file)

        # Identify the probability column (exclude id/target)
        ignore = {'id', 'target', TARGET}
        prob_cols = [c for c in df.columns if c not in ignore]
        
        # Rename columns to include model name to avoid collisions
        # e.g., 'catboost_prob_low', 'catboost_prob_medium', etc.
        rename_map = {c: f"{model_name}_{c}" for c in prob_cols}

        # Set index and keep only the renamed prob columns
        subset = df.set_index(index_col).rename(columns=rename_map)[list(rename_map.values())]
        df_list.append(subset)

    # Join all models side-by-side
    return pd.concat(df_list, axis=1)

In [7]:
# Create DataFrames
oof_df = load_probs(oof_files)
test_df = load_probs(test_files)

# Look for missing rows in test data
missing_by_model = test_df.isna().sum().sort_values(ascending=False)
print("Missing test rows per model:")
print(missing_by_model[missing_by_model > 0])

if missing_by_model.any():
    bad = missing_by_model[missing_by_model > 0].index.tolist()
    print("\nFirst few missing ids for each bad model:")
    for m in bad:
        print(m, test_df.index[test_df[m].isna()][:10].tolist())

# Create an id-indexed label series (0/1)
y_by_id = pd.Series(y_true.values, index=training_df['id'].values)

# Keep only rows where all models have OOF probabilities
oof_df = oof_df.sort_index().dropna(axis=0)
y_true_aligned = y_by_id.loc[oof_df.index].values

# Align test rows by id as well
test_df = test_df.sort_index()

print(f'\nOOF Shape: {oof_df.shape}')
print(f'Test Shape: {test_df.shape}')
print(f'\nCorrelation: \n{oof_df.corr()}')

Missing test rows per model:
Series([], dtype: int64)

OOF Shape: (577347, 12)
Test Shape: (247435, 12)

Correlation: 
                               catboost_prob_qso  catboost_prob_star  \
catboost_prob_qso                          1.000              -0.242   
catboost_prob_star                        -0.242               1.000   
catboost_prob_galaxy                      -0.682              -0.545   
lgb_prob_qso                               0.998              -0.240   
lgb_prob_star                             -0.239               0.996   
lgb_prob_galaxy                           -0.682              -0.541   
nn_tabular_resnet_prob_qso                 0.993              -0.269   
nn_tabular_resnet_prob_star               -0.292               0.980   
nn_tabular_resnet_prob_galaxy             -0.676              -0.538   
xgb_prob_qso                               0.998              -0.241   
xgb_prob_star                             -0.240               0.996   
xgb_prob_galaxy  

In [8]:
print('\nIndividual Model Balanced Accuracy (OOF):')
individual_scores = {}

# Group columns by model to evaluate them
model_names = list(set([c.split('_prob_')[0] for c in oof_df.columns]))

for model in model_names:
    cols = [f"{model}_prob_qso", f"{model}_prob_star", f"{model}_prob_galaxy"]
    # Get hard labels via argmax
    preds = np.argmax(oof_df[cols].values, axis=1)
    score = balanced_accuracy_score(y_true_aligned, preds)
    individual_scores[model] = score
    print(f'{model:10}: {score:.6f}')

best_single_model = max(individual_scores, key=individual_scores.get)
print(f'\nBest single model: {best_single_model} ({individual_scores[best_single_model]:.6f})')


Individual Model Balanced Accuracy (OOF):
nn_tabular_resnet: 0.956917
xgb       : 0.965734
catboost  : 0.964935
lgb       : 0.965587

Best single model: xgb (0.965734)


## Stacking Optimization: Logistic Regression Meta-Model

We train a second-level meta-model on the base models' **OOF probabilities**.

* **Features (X):** OOF probabilities from each base model.
* **Target (y):** true class label (0=QSO, 1=STAR, 2=GALAXY).
* 
We use **logistic regression with L2 regularization** as the meta-model. It is fast, stable under correlation, and outputs a valid probability in [0, 1].
The regularization strength is tuned via internal cross-validation.


In [9]:
# ==========================================
# STACKING STRATEGY (Meta-Model)
# ==========================================

print('Preparing stacking data...')

model_cols = list(oof_df.columns)
print('Stacking models:', model_cols)

X_stack = oof_df[model_cols].values
y_stack = y_true_aligned
X_test_stack = test_df.values

# Standardize meta-features (helps logistic regression)
scaler = StandardScaler()
X_stack_s = scaler.fit_transform(X_stack)
X_test_stack_s = scaler.transform(X_test_stack)

# Multi-class Logistic Regression Stacker
meta_model = LogisticRegressionCV(
    Cs=20,
    cv=5,
    scoring='balanced_accuracy',
    penalty='l2',
    max_iter=5000,
    n_jobs=-1,
    random_state=seed,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

# True OOF predictions from the meta-model
oof_meta_preds = cross_val_predict(
    meta_model, X_stack_s, y_stack,
    cv=cv,
    method='predict'
)

score_meta = balanced_accuracy_score(y_stack, oof_meta_preds)
print(f'Meta-model OOF Balanced Accuracy: {score_meta:.6f}')

meta_model.fit(X_stack_s, y_stack)

print(f'\nCoefficients: \n{meta_model.coef_}')

Preparing stacking data...
Stacking models: ['catboost_prob_qso', 'catboost_prob_star', 'catboost_prob_galaxy', 'lgb_prob_qso', 'lgb_prob_star', 'lgb_prob_galaxy', 'nn_tabular_resnet_prob_qso', 'nn_tabular_resnet_prob_star', 'nn_tabular_resnet_prob_galaxy', 'xgb_prob_qso', 'xgb_prob_star', 'xgb_prob_galaxy']
Meta-model OOF Balanced Accuracy: 0.962879

Coefficients: 
[[ 4.20825632e-01 -1.67664935e-01 -2.37270216e-01  3.85827492e-01
  -2.66550751e-01 -1.32685292e-01  3.00294295e-01 -6.15562633e-02
  -2.25818509e-01  4.17448686e-01 -1.96799980e-01 -2.12393581e-01]
 [-1.67940630e-01  3.10364799e-01 -8.88870655e-02 -1.75989151e-01
   4.71126743e-01 -2.02335844e-01 -2.99517699e-01  6.11689257e-02
   2.25417245e-01 -1.16185235e-01  5.18852880e-01 -2.90167954e-01]
 [-2.52885002e-01 -1.42699864e-01  3.26157281e-01 -2.09838341e-01
  -2.04575991e-01  3.35021136e-01 -7.76596160e-04  3.87337584e-04
   4.01264291e-04 -3.01263451e-01 -3.22052900e-01  5.02561535e-01]]


## Final Ensemble & Submission

We generate test-set probabilities by applying the trained meta-model to the matrix of base-model test probabilities.
The submission file uses the competition's sample submission schema.


In [10]:
# Fill submission using sample_submission column name
sub = submission_df.copy()
target_col = [c for c in sub.columns if c != 'id'][0]
reverse_mapping = {v: k for k, v in target_mapping.items()}

# Generate test probabilities from the stacker
test_meta_probs = meta_model.predict_proba(X_test_stack_s)   # (N, 3) matrix
test_meta_pred = np.argmax(test_meta_probs, axis=1)

sub[target_col] = pd.Series(test_meta_pred).map(reverse_mapping)

sub.to_csv('submission.csv', index=False)
print('Saved: submission.csv')

print('\nSUBMISSION')
print('==========')
print(sub.head(10))

print('\nPredicted Class Distribution:')
print(sub[target_col].value_counts(normalize=True).sort_index())

Saved: submission.csv

SUBMISSION
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY
5  577352  GALAXY
6  577353  GALAXY
7  577354    STAR
8  577355  GALAXY
9  577356  GALAXY

Predicted Class Distribution:
class
GALAXY   0.646
QSO      0.205
STAR     0.149
Name: proportion, dtype: float64
